# Why this graph exists

The co-occurrence chart in `assets/charts/character_cooccurrence.png` counts substrings. That chart is still the right tool for a fast lexical look, and this notebook does not change it.

Three facts in *Dracula* make substring counts a weak stand-in for people and places:

- Mina writes first as Mina Murray and later as Mina Harker. Both strings are one person. The narrator field keeps both labels.
- The string `Dracula` sits inside Castle Dracula and Count Dracula. One is a building. One is a person.
- Arthur Holmwood becomes Lord Godalming. The surname changes, and a title appears.

The rest of these notebooks build an entity-resolution graph on the corpus that is already in this repo: `data/dracula_ascii_rag.json`. Chunks overlap by about 150 characters. Mention offsets use a de-overlapped entry string. The RAG index keeps the overlap.

Library pins live in `requirements.txt` (`spacy>=3.8`, `rapidfuzz>=3.0`). The next cell prints the versions installed in this environment. This walkthrough was checked with spaCy 3.8 and rapidfuzz 3.14.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "kg").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(ROOT)


In [ ]:
import importlib.metadata as metadata
from collections import Counter

from src.kg.entries import load_entries

for name in ("spacy", "rapidfuzz", "pandas", "pyyaml", "matplotlib"):
    print(f"{name} {metadata.version(name)}")

entries = load_entries(ROOT / "data" / "dracula_ascii_rag.json")
print(len(entries), "entries")
print("chapters", min(e.chapter_number for e in entries), "to", max(e.chapter_number for e in entries))
print(Counter(e.narrator for e in entries))


## What the later notebooks check

A useful v1 graph keeps `castle_dracula` and `count_dracula` as different ids, sends both Mina narrator strings to `mina_harker`, and scores proper-name pairs on chapters 14–27. Chapters 1–13 are the tune split. The gold alias file is knowledge: the held-out score is reported with that file and without it.

Pattern relations are strict on purpose. A kinship word has to sit in the short gap between two names. The novel usually writes "married to Jonathan" beside one name, so the pattern layer can emit nothing. A second layer checks relationships known from the book (`data/gold/relations.yml`) against the text and keeps only those a sentence supports, with that sentence as evidence. Carfax as part of London is a curated edge in `configs/kg.yml`.
